# Business Analysis Group Project
#### **Dataset**: [Open e-commerce 1.0: Five years of crowdsourced U.S. Amazon purchase histories with user demographics](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YGLYDY)

## Business Problem


**BUSINESS PROBLEM PROPOSAL**

How do demographic factors like age, income, education constitute the highest-value customers and which geographic locations (on a state level) drive customer value and dictate regional product demand, among U.S. Amazon customers?
Are there regional variations in product category preferences across U.S. states?


## Objectives

- Analyze sales performance and customer behaviour based on demography
- Identify top-selling products in different geographic locations (and high-value customers)
- Examine relationships between quantity, pricing and revenue

## Proposed Analysis

- Sales trend analysis
- Product and customer analysis
- Revenue analysis

## Analysis Roadmap

### Data Integration
- [x] Import relevant libraries
- [x] Load `amazon-purchases.csv` and `survey.csv` into DataFrames


### Data Cleaning and Preprocessing

- [x] Type Casting, Missing Values, Outliers
- [x] **Data Integration**: Merge the purchase data with the demographic data on the `Survey ResponseID` key to create a master dataset.
- [x] Data Reduction, Transformation


### Preliminary Analysis (Summary Statistics)
- [x] High level view of data using `.describe()` and `.info()`
- [x] Extract interesting metrics

### Exploratory Data Analysis

- [x] **Univariate Analysis**:
    - Distribution of `Total Price`
    - Frequency counts for `Category`, `Agg_Category`, `Q-demos-age`, `Q-demos-income`, `Q-demos-state`
- [x] **Multivariate Analysis**:
    - Revenue distribution by `Income Group`
    - Revenue distribution by `Age Group`
    - Revenue distribution by `State` (`Q-demos-state`)
    - Monthly Revenue Trends
    - Daily Revenue Patterns by `Day of Week`


### Summary and Results
- [x] Present the results using visualizations (`matplotlib`).
Bar plots (seaborn), histograms, line graphs have been used, In time patterns matplotlib is used*
- [x] Derive business insishgts from descriptive patterns and correlations.
- [x] Create Dashboard from findings


### Expected Outcome (Future)


- Improve inventory planning
- Target profitable customer groups
- Optimize product offerings
- Increase overall revenue


### Research Questions (Future)
1. What are the customer's spending trends over time?
Monthly and yearly spending patterns.
    
2. Which product categories generate the most revenue?
Top categories by total spend.
    
3. Which categories are purchased most frequently?
Top categories by number of orders.
    
4. Are purchases concentrated in a few categories or spread across many?
Use a Pareto chart or category share chart.
    
5. What products are repeatedly purchased?
Use ASIN/product code frequency.
    
6. Are there unusual high-value purchases?
Identify outliers such as expensive electronics or consoles.
Which product categories in which state should be recommended?



---



## Discussion

We could also add another preprocessed Dataset including a mapping between the ISBN codes and the book categories. The dataset is available here:
https://github.com/aberke/amazon-study/blob/master/data-analysis/preprocessed/google_books_api_isbn_category.csv

# Data Integration

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# The datasets need to be placed in a directory called 'Business Analytics' on Google Drive. Arrange it according your files location on Google Drive

# drive_directory = '/content/drive/MyDrive/Business Analytics'

drive_directory = '/content/drive/MyDrive/Business Analytics'

amzn_purchases_csv = os.path.join(drive_directory, 'amazon-purchases.csv')
survey_csv = os.path.join(drive_directory, 'survey.csv')

# Save the DataFrame to CSV
amzn_data = pd.read_csv(amzn_purchases_csv)
survey_data = pd.read_csv(survey_csv)


# Data Cleaning and Preprocessing

In [ ]:
# Merge Data
data = pd.merge(amzn_data, survey_data, left_on='Survey ResponseID', right_on='Survey ResponseID')

In [ ]:
# List of columns to be removed
columns_to_remove = [
    'Q-substance-use-cigarettes',
    'Q-substance-use-marijuana',
    'Q-substance-use-alcohol',
    'Q-personal-diabetes',
    'Q-demos-race',
    'Q-amazon-use-howmany',
    'Q-amazon-use-hh-size',
    'Q-personal-wheelchair',
    'Q-life-changes',
    'Q-sell-YOUR-data',
    'Q-sell-consumer-data',
    'Q-small-biz-use',
    'Q-census-use',
    'Q-research-society',
    'Q-sexual-orientation'
]

# Drop the specified columns
data = data.drop(columns=columns_to_remove, errors='ignore')

In [ ]:
# Type Casting and Data Cleaning
# Convert 'Order Date' to datetime
data['Order Date'] = pd.to_datetime(data['Order Date'], errors='coerce')

# Convert 'Purchase Price Per Unit' and 'Quantity' to numeric
data['Purchase Price Per Unit'] = pd.to_numeric(data['Purchase Price Per Unit'], errors='coerce')
data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce')

# Convert 'Q-demos-hispanic' to boolean
data['Q-demos-hispanic'] = (
    data['Q-demos-hispanic']
    .astype(str)
    .str.lower()
    .str.strip()
    .map({'yes': True, 'no': False})
    .astype('boolean')
)

# Convert other columns to categories
for col in ['Shipping Address State', 'Category','Q-demos-age', 'Q-demos-education', 'Q-demos-income', 'Q-demos-gender', "Q-demos-state", 'Q-amazon-use-how-oft']:
    data[col] = data[col].astype('category')

print("DEBUG: Data types after initial conversion:")
display(data.info())

### Data Transformations

We will perform the following transformations:

1.  **Extract Date Components**: Create new columns for year, month, and day of the week from the 'Order Date' column.
2.  **Calculate Total Price**: Create a 'Total Price' column by multiplying 'Purchase Price Per Unit' and 'Quantity'.
3. **Aggregate Categories**: Create an 'Agg_Category' column by trying to classify our default categories with known search terms

In [ ]:
# Extract date components
data['Order Year'] = data['Order Date'].dt.year
data['Order Month'] = data['Order Date'].dt.month
data['Order DayOfWeek'] = data['Order Date'].dt.dayofweek # Monday=0, Sunday=6

# Calculate Total Price
data['Total Price'] = data['Purchase Price Per Unit'] * data['Quantity']

print("DataFrame after adding 'Order Year', 'Order Month', 'Order DayOfWeek', and 'Total Price' columns:")
display(data.head())

In [ ]:
# Aggregate Category data into more generic groups and add them to another column

# Load a map of predefined General-Categories identified by search (sub-)strings
mapping_df = pd.read_csv(os.path.join(drive_directory, 'category_map.csv'))

conditions = []
choices = []

# Build the rule set
for index, row in mapping_df.iterrows():
  # Create the search condition (case-insensitive, ignore nan)
  condition = data['Category'].str.contains(row['SEARCH_STRINGS'], case=False, na=False)
  conditions.append(condition)
  choices.append(row['NEW_CATEGORY'])

# Apply the rules and
data['Agg_Category'] = np.select(conditions, choices, default='Other / Misc')
data['Agg_Category'] = data['Agg_Category'].astype('category')
print(data['Agg_Category'].value_counts())


# Extract distinct 'Category' fields from the 'Other / Misc' group for inspection of missed out groups
misc_data = data[data['Agg_Category'] == 'Other / Misc']
unique_misc_categories = misc_data['Category'].dropna().unique()
unique_misc_categories_df = pd.DataFrame(unique_misc_categories, columns=['Category'])
unique_misc_categories_df.to_csv('unique_misc_categories.csv', index=False)
print("unique_misc_categories.csv has been saved for inspection.")

In [ ]:
print("Info after transformations:")
display(data.info())

# Preliminary Analysis (Summary Statistics)

In [ ]:
# Missing value info
#print("Missing values per column:")
#display(data.isnull().sum())

# Remove Missing values from 'Category' since this information will be used for the main analysis
data = data.dropna(subset=['Category'])

# Everything else can stay since Title, since those are not important for analysis
# In the case of 'Shipping Address State' we can just use 'Q-demos-state' which has no missing values
# But we should make sure that this can be used for correlating the datat
# Note: If we decide to create a Category to ISBN mapping (see Discussion) we could drop those as well

print("Missing values after elimination:")
display(data.isnull().sum())

In [ ]:
# Inspect data

print("=== ALL DATA TYPES ===")
print(data.columns)
print("\n")

print("=== CATEGORIES ===")
for col in data.select_dtypes(include=['category']).columns:
  print(data[col].value_counts(dropna=False))
  print("\n")

print(data['Q-demos-hispanic'].value_counts(dropna=False))


# Save categories to csv to make inspecting easier.
category_counts = data['Category'].value_counts().reset_index()
category_counts.columns = ['Category', 'Count']
category_counts.to_csv('category_counts.csv', index=False)

print("q_demos_hispanic_counts.csv has been saved.")

# To inspect categories in code we can use one of the following access functions
# Using the .cat.categories attribute
# display(data['Category'].cat.categories)

# Using the .unique() method
# display(data['Category'].unique())

In [ ]:
print("=== OVERALL DATA SUMMARY")
display(data.info())

# Disable scientific notation for displaying the data
pd.set_option('display.float_format', lambda x: '%.2f' % x)

display(data.select_dtypes(include=['int64', 'float64', 'datetime64']).describe())
display(data.select_dtypes(include=['category', 'boolean', 'object']).describe())

# Alternatively display all columns
# display(data.describe(include='all'))

# Enable scientific notation again
pd.reset_option('display.float_format')

print("\n First rows:")
display(data.head())


### Aggregated Views

May add more of those based on analysis needs

In [ ]:
# Aggregate total sales by Category
sales_by_category = data.groupby('Category')['Total Price'].sum().sort_values(ascending=False)
print("         Top 10 Categories by Total Sales:")
display(sales_by_category.head(10))

In [ ]:
# Aggregate total sales by Shipping Address State
sales_by_state = data.groupby('Shipping Address State')['Total Price'].sum().sort_values(ascending=False)
print("Top 10 States by Total Sales:")
display(sales_by_state.head(10))

In [ ]:
# Aggregate total sales by Q-demos-income
sales_by_income = data.groupby('Q-demos-income')['Total Price'].sum().sort_values(ascending=False)
print("Total Sales by Income Level:")
display(sales_by_income)



---



# Exploratory Data Analysis and Descriptive Statistics

Now, here we explore the cleaned and merged Amazon purchase and survey dataset. We find the descriptive patterns that help us tackle the business problem we defined and also explain the customers types' and product categories' which are contributing in the Amazon revenues.

**Initial data exploration**

In [ ]:
data_shape = data.shape # dimensions (rows, columns) of data
column_names = data.columns.tolist() # all column names in data as a list
missing_values = data.isnull().sum().sort_values(ascending=False).head(15) # top 15 columns with the most missing values in data ; no. of missing values for each column
duplicate_rows_count = data.duplicated().sum() # total number of duplicate rows in data
data_types = data.dtypes # data type for each column

print(f"\nShape: {data_shape}")
print(f"\nColumns:\n {column_names}")
print(f"\nMissing values (top 15):\n {missing_values}\nQ-demos-state has no missing values and can be used for correlating data. Q-demos-state is sufficient for our geographic analysis. Thus, we consider the missing Shipping Address State less critical.")
print(f"\nDuplicate rows: {duplicate_rows_count}\nIn the purchase dataset here we see that some rows are duplicates because we assume they might represent actual, distinct purchases by the same customer.")
print(f"\nData types:\n {data_types} \ndata is in the correct format for analysis")

This overview confirms that the dataset is already well prepared for analysis, with key variables for time, revenue, category, geography, and demographics. That means we can focus on patterns and relationships instead of basic cleaning.

**Descriptive Statistics for Numerical Variables**

We see now the statistical summary for numerical features like `Purchase Price Per Unit`, `Quantity`, `Total Price`, `Order Year`, and `Order Month`.

In [ ]:
display(data[["Purchase Price Per Unit", "Quantity", "Total Price", "Order Year", "Order Month"]].describe())

**Value Counts for Categorical Variables**

So, let's check now the distribution of categorical variables:- top 10 most frequent values each category. We get to know the dominant products, their categories and customer demographics.

In [ ]:
amz_cols = ["Category", "Agg_Category", "Q-demos-age", "Q-demos-income", "Q-demos-gender", "Q-demos-state", "Q-amazon-use-how-oft"]

for col in amz_cols:
    print(f"\n    {col} (Top 10 Values) ") # selecting the top 10 most frequent values and their counts.
    display(data[col].value_counts().head(10)) # For each categorical column, counting the occurrences of each unique value.


---



##Revenue distribution
**Histogram to visualize the distribution of the Total Price in our dataset**

In [ ]:
plt.figure(figsize=(12, 6)) # figure is 12 inches wide and 6 inches tall
sns.histplot(data["Total Price"], bins=50, kde=True, color="darkcyan")# Plotting Total Price from our data
# 50 equal intervals for the Total Price and no. of purchases in each one ; KDE curve in the histogram giving continuous view
plt.title("Distribution of Total Price")
plt.xlabel("Total Price")
plt.ylabel("Number of Purchases") # the frequency count for each price bin
plt.xlim(0, data["Total Price"].quantile(0.99))
# 99% percentile will show 99% of our data-> distribution clearer and no large outliers.
plt.tight_layout()
plt.show()




*   The plot's x-axis (Total Price) is indeed showing values up to around $175. This is to show only the transactions up to the 99th percentile of Total Price.
* Products costing thousands of dollars are present in our data, but they are outliers, and here our plot focuses on the bulk/most of the purchases.
*   We see 1e6 on the y-axis. So, the highest no. of purchases in our graph is two million.
*   The highest bar in the histogram- large number of purchases around approx. 1.75 million are those of products costing low.



**This chart shows** that *most purchases have low total values*, while a small number of very large orders stretch the distribution to the right. That means *revenue is skewed*, so the business should pay attention to high-value transactions rather than only average purchase size.

**The reason**- Focusing purely on the average would be misleading because those few large orders contribute disproportionately to the total revenue.

**Now, check here the Main distribution while still acknowledging outliers**, *which also by the way confirms that high-value transactions exist but the vast majority of purchases are for lower amounts* :-

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.histplot(data["Total Price"], bins=50, kde=True, color="darkcyan")
plt.title("Distribution of Total Price")
plt.xlabel("Total Price $")
plt.ylabel("Number of Purchases in Millions")


plt.tight_layout()
plt.show()


# price range bins: $250 buckets
bin_edges = np.arange(0, data["Total Price"].max() + 250, 250)
counts, edges = np.histogram(data["Total Price"], bins=bin_edges)
bin_centers = (edges[:-1] + edges[1:]) / 2

# Calculate the sum of 'Total Price' for each bin
# Use pd.cut to group data into the same bins as the histogram
binned_total_price = pd.cut(data["Total Price"], bins=bin_edges, include_lowest=True)
revenue_per_bin = data.groupby(binned_total_price)["Total Price"].sum().values

fig, ax = plt.subplots(figsize=(15, 8))

# Bars : counts per $250 bucket
ax.bar(bin_centers, counts, width=220, color="darkcyan", alpha=0.55,
       edgecolor="white", label="Bar indicate Purchase counts log value")

# Line :  connecting the bar tops
ax.plot(bin_centers, counts, color="black", linewidth=2, marker="o",
        markersize=4, label="Line indicate Purchase counts value")

# Log scale y-axis : low counts at high prices -> visible
ax.set_yscale("log")
# Set x-ticks at the center of each bar
ax.set_xticks(bin_centers)
# Create labels for each bin center, showing the price range for that bin
bin_labels = [f'${edges[i]:.0f}-${edges[i+1]:.0f}' for i in range(len(bin_centers))]
ax.set_xticklabels(bin_labels, rotation=90, ha='right', fontsize=8)
ax.set_xlabel("Total Price $")
ax.set_ylabel("Number of Purchases in log scale")
ax.set_title("Distribution of Total Price")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# Annotate non-zero bars with both count and total revenue
for i, (c, ct) in enumerate(zip(bin_centers, counts)):
    if ct > 0:
        # Format total revenue for display
        rev_text = f'${revenue_per_bin[i]:,.0f}'
        ax.annotate(f"Count: {ct:,}\nRev: {rev_text}", (c, ct), textcoords="offset points",
                    xytext=(0, 8), fontsize=7, rotation=45, ha="left")

plt.tight_layout()
plt.show()

##Top categories by purchase count

**We now see bar plot to visualize the top 10 most frequently purchased product categories in our dataset.**

In [ ]:
top_amz_count = data["Category"].value_counts().head(10)

plt.figure(figsize=(12, 10))
sns.barplot(
    x=top_amz_count.values,
    y=top_amz_count.index.astype(str),
    palette="pastel",
    hue=top_amz_count.index.astype(str),
    legend=False
)
#  x-axis=counts of purchases for each category. y-axis=names of the categories
plt.title("Most purchased 10 Categories")
plt.xlabel("Purchases")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

The `Category` column used for above **bar plot** has long and numerous *category names which overlap* and thus may not represent correct and unique information about top purchases.


1.   We created `Agg_Category` column in **data preparation** phase- to *group similar* `Category` names into more concise categories.
2.   This *helps analyzing top categories and greatly reduce label overlap*



In [ ]:
top_amz_count = data["Agg_Category"].value_counts().head(10)

plt.figure(figsize=(12, 10))
sns.barplot(
    x=top_amz_count.values,
    y=top_amz_count.index.astype(str),
    palette="pastel",
    hue=top_amz_count.index.astype(str),
    legend=False
)

#  x-axis represents the counts of purchases for each category. y-axis display the names of the categories
plt.title("Top 10 Categories by Purchase Count")
plt.xlabel("Number of Purchases")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

This chart shows which categories appear most often in customer purchases. These are likely the most visible categories in customer behavior and should be prioritized in stock planning, merchandising, and promotions.

In [ ]:
# Checking the largest 'Other / Misc' aggregated category
unique_misc_categories_counts_df = data[data['Agg_Category'] == 'Other / Misc']['Category'].value_counts()

print("       No. of purchases of Top 10 categories in 'Other / Misc'")
display(unique_misc_categories_counts_df.head(10))

Lets us see more **insights** from the **graph** above:-


*   Firstly, the **largest category is '*Other / Misc*'**. Many diverse and less frequent categories are grouped into it, thereby artificially increasing count.
*   Thus, for our consideration-  **'*Groceries & Food*' is the highest sold category** with over 200k ourchases.



##Demographic patterns



1.   **Revenue by Income group**
2.   **Revenue by Age group**
3.   **Revenue by State**



### Revenue by Income group

In [ ]:
rev_incom = data.groupby("Q-demos-income")["Total Price"].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=rev_incom.values, y=rev_incom.index, palette="Set2")
plt.title("Revenue by Income Group")
plt.xlabel("Total Revenue in Millions")
plt.ylabel("Income Group")
plt.tight_layout()
plt.show() # 1e6 = 1 * 10^6

This **chart shows** *revenue distribution* across income groups. It tells us whether higher-income or lower-income customers contribute more spending, which is useful for promotions basis- *income segmentations*.


*   The **highest revenue** is generated from the group **100k - 150k** (excluding) = more than 8 million



###Revenue by Age group

In [ ]:
age_wise_rev = data.groupby("Q-demos-age")["Total Price"].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=age_wise_rev.values, y=age_wise_rev.index, palette="Set2")
plt.title("Revenue by Age Group")
plt.xlabel("Total Revenue in 10 Millions")
plt.ylabel("Age Group")
plt.tight_layout()
plt.show() # 1e7 is 1 * 10^7

This **chart shows** which **age groups** contribute the most revenue. **Age-based segmenting** during promotions and can help match *product recommendations* and also put pricing more effectively.


*   The **age goup 25 to 34 years** generate *most revenue*, more than around **13 million**.



###Revenue by State

In [ ]:
# 'Q-demos-state'
revenue_state = data.groupby("Q-demos-state")["Total Price"].sum().sort_values(ascending=False)

plt.figure(figsize=(14, 15))
sns.barplot(x=revenue_state.values, y=revenue_state.index, palette="viridis")
plt.title("Revenue by State")
plt.xlabel("Total Revenue in Millions")
plt.ylabel("State")
plt.tight_layout()
plt.show()

This **chart shows which states generate the highest sales**. Demand is different geographically , hence for **logisitcs purposes and marketing**, there must be *different planning for each state*- prioritizing *strongest ones first*!


*   The **highest** or strongest **revenue** is from the state of **California** around more than **4 million** dollars.




#Time patterns


1.   **Monthly trend**

2.   **Days in week pattern**



###Monthly trend

In [ ]:
import matplotlib.dates as mdates

revenue_monthwise = data.groupby(["Order Year", "Order Month"])["Total Price"].sum().reset_index()
revenue_monthwise["YearandMonth"] = pd.to_datetime(revenue_monthwise["Order Year"].astype(str) + "-" + revenue_monthwise["Order Month"].astype(str) + "-01")

revenue_monthwise_recorded = revenue_monthwise[revenue_monthwise['Total Price'] > 0] # zero revenue is removed at the end

plt.figure(figsize=(14, 6))
sns.lineplot(data=revenue_monthwise_recorded, x="YearandMonth", y="Total Price", color="orange")
plt.title("Monthly Revenue Trend")
plt.xlabel("Dates with Year-Month")
plt.ylabel("Total Revenue in Millions")

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3)) # inteveral of 3 months shown in graph
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

month_last_recorded = revenue_monthwise_recorded['YearandMonth'].max()
print(f"Last recorded month in the data : {month_last_recorded.strftime('%Y-%m')}")


**Key Insight and Observation**: The revenue drops sharply to near zero around 2023. Sales drastically decreased.


*   This chart shows how the revenue rises and falls over time and when are the peaks. **Amazon** can **manage inventory and marketing ads** accordingly and consider *steps to take during  high demand periods*.



###Days in week pattern

In [ ]:
daysorder = [0, 1, 2, 3, 4, 5, 6] # represents days of week respectively
weekrevenue = data.groupby("Order DayOfWeek")["Total Price"].sum().reindex(daysorder) # day of the week when an order was placed

days_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'] # actual day names
weekrevenue_day = weekrevenue.rename(index=lambda x: days_week[x])

plt.figure(figsize=(10, 5))
sns.barplot(x=weekrevenue_day.index, y=weekrevenue_day.values, palette="Set2") # show the names of the days on the x-axis
plt.title("Revenue each Day of Week")
plt.xlabel("Days")
plt.ylabel("Total Revenue in millions")
plt.tight_layout()
plt.show()

The plot shows the total revenue for **each day of the week**, summed **across all the years** in our data.


*   This *chart shows how buying behavior changes across the week*. Amazon can time their campaigns and promotions for those *days of the week when customers are most active*.
*  The *highest sales/revenue is on **mondays*** which is more than around *6 million*.



#Outliers
###Boxplot

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=data["Total Price"])
plt.title("Total price BoxPlot")
plt.xlabel("Total Price")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 10))
sns.boxplot(x=data["Total Price"])
plt.title("Total Price BoxPlot with Outlier")
plt.xlabel("Total Price $")

# Quartiles Q1, Q3, and Inter Quartile Range IQR
Q1 = data["Total Price"].quantile(0.25)
Q3 = data["Total Price"].quantile(0.75)
IQR = Q3 - Q1

# Upper bound for outliers 1.5 * IQR
upperbound = Q3 + 1.5 * IQR

# 99th percentile
percentile99th = data["Total Price"].quantile(0.99)

plt.xlim(0, percentile99th * 1.1) # Show up to 110% of the 99th percentile

# Indicate the upper bound of purchases using IQR
plt.axvline(x=upperbound, color='red', linestyle='--', label=f'Outliers beyond Q3 + 1.5*IQR = ${upperbound:.2f}')

# Indicate 99th percentile
plt.axvline(x=percentile99th, color='blue', linestyle=':', label=f'99th Percentile = ${percentile99th:.2f}')

# Add text annotations for interpretation
plt.text(upperbound + 5, plt.ylim()[1] * 0.8, 'Outliers', color='red', ha='left', va='center', fontsize=10)
plt.text(percentile99th + 5, plt.ylim()[1] * 0.7, 'Very High Value Purchases', color='blue', ha='left', va='center', fontsize=10)
plt.text(Q1, plt.ylim()[1] * 0.9, 'Most Purchases', color='black', ha='left', va='center', fontsize=10)

plt.legend()
plt.tight_layout()
plt.show()

### Boxplot: Total Price Distribution

The  visualization depicts the highly skewed nature of 'Total Price' distribution:

*   **Right-Skewed Distribution**: The median is closer to the left side of the box. Majority of purchases are for smaller amounts. A few large ones pull the mean to the right. Thus there are many small values and few large ones.
*   **Outliers**: Purchases more than **`$48.82`** (`Q3 + 1.5 * IQR`) are outliers, significantly different from the majority.
*   **Very High-Value Purchase**: The **99th percentile at `$180.46`** highlights high-value transactions. About 1% of purchases exceed this making rare ones!


**Key Observation**: *Amazon should not ignore high-value customers because they can contribute a disproportionate share of revenue.*


---



## Business Insights

From the EDA - exploratory data analysis done above, we can infer/derive very valuable **business insights**:

*   **Revenue Distribution**: The **`Total Price`** distribution is *highly skewed*, having *many low value transactions* and a very few high-value ones which make disproportionate contribution to overall revenue. *Inexpensive items give the most volume*. Still *focusing on high-value customers and transactions* is in a way important for *maximizing total revenue*.

*   **Product Categories Sold**:
    *   In original `Category` column, **'ABIS_BOOK'** is the most purchased category item.
    *   In the aggregated **`Agg_Category`** column, **'Groceries & Food'** is most purchased category.
    *   From this we *inform* ourselves of **inventory planning, specific promotions, advertisements and product development/management strategies**.

*   **Income Group Contribution**: The income group **$100,000 - $149,999** generates the highest total revenue. Specific income groups are most valuable and can be focused upon and sending them specific marketing campaigns with product offerings and advertisements.

*   **Age Group Contribution**: Customers in the **25 - 34 years** age group contribute the most revenue, very ahead of others. Age-based market categorisation, product recommendations, and advertisements can be very useful.

*   **Geographic Revenue Contribution**: **California** state generates the highest total revenue. We have therefore regional differences in demand, thus giving us knowledge of logistics, and localized marketing.

*   **Monthly and Daily Revenue Trends**:
    * There is a *sharp decline* around **2023** and *sharp increase* at round **end of 2021**. Amazon can identify monthly periods of very high or demand and manage inventory, plan marketing campaigns.
    * Similarly in **daily pattern**, **Mondays** generate *the highest revenue* from any other day of the week. This tells, customers are *mostly active at the beginning of the week* for purchase on Amazon. This then is very useful to correctly time promotions and allocate resources.


**Dashboard**

In [ ]:
# @title Amazon Purchase Strategic Dashboard {display-mode: "form"}

import pandas as pd
import json
from IPython.display import HTML, display

try:
    from google.colab import output
except Exception:
    output = None

def get_dashboard_data():
    global data
    month_names = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
    day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    available_years = sorted(data["Order Year"].dropna().astype(int).unique().tolist())
    available_states = sorted(data["Q-demos-state"].dropna().astype(str).unique().tolist())

    year_month_map = {}
    for yr in available_years:
        months = sorted(data.loc[data["Order Year"].astype(int) == yr, "Order Month"].dropna().astype(int).unique().tolist())
        year_month_map[int(yr)] = [{"val": int(m), "name": month_names.get(int(m), str(m))} for m in months]

    filter_cols = ["Order Year", "Order Month", "Q-demos-state"]

    # KPIs
    kpi_data = data.groupby(filter_cols, observed=True).agg({"Total Price": ["sum", "mean"], "Survey ResponseID": "nunique"}).reset_index()
    kpi_data.columns = ["year", "month", "state", "rev", "aov", "users"]
    kpi_data["orders"] = data.groupby(filter_cols, observed=True).size().values

    # Trend
    trend_data = data.groupby(["Order Year", "Order Month", "Q-demos-state"], observed=True)["Total Price"].sum().reset_index()
    trend_data.columns = ["year", "month", "state", "rev"]

    # Day of Week
    dow_data = data.groupby(filter_cols + ["Order DayOfWeek"], observed=True)["Total Price"].sum().reset_index()
    dow_data.columns = ["year", "month", "state", "dow", "rev"]

    # Categories
    temp_df = data.copy()
    temp_df["Display_Category"] = temp_df["Agg_Category"].astype(str)
    mask = temp_df["Display_Category"] == "Other / Misc"
    temp_df.loc[mask, "Display_Category"] = temp_df.loc[mask, "Category"].astype(str)
    cat_data = temp_df.groupby(filter_cols + ["Display_Category"], observed=True).size().reset_index(name="count")
    cat_data.columns = ["year", "month", "state", "Display_Category", "count"]

    # Age
    age_data = data.groupby(filter_cols + ["Q-demos-age"], observed=True)["Total Price"].sum().reset_index()
    age_data.columns = ["year", "month", "state", "Q-demos-age", "Total Price"]

    return {
        "years": available_years,
        "states": available_states,
        "month_labels": [month_names[i] for i in range(1, 13)],
        "day_labels": day_names,
        "year_month_map": year_month_map,
        "overall_kpis": {
            "total_revenue": float(data["Total Price"].sum()),
            "total_orders": int(len(data)),
            "avg_order_value": float(data["Total Price"].mean()),
            "unique_customers": int(data["Survey ResponseID"].nunique())
        },
        "kpi_map": kpi_data.to_dict("records"),
        "trend_map": trend_data.to_dict("records"),
        "dow_map": dow_data.to_dict("records"),
        "categories": cat_data.to_dict("records"),
        "age": age_data.to_dict("records")
    }

try:
    dashboard_json = json.dumps(get_dashboard_data())
except Exception as e:
    dashboard_json = json.dumps({"error": f"Data error: {str(e)}"})

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap" rel="stylesheet">
<style>
    :root { --bg: #f7fafc; --card: #ffffff; --border: #e2e8f0; --text: #1a202c; --muted: #718096; --primary: #4c51bf; --bar-bg: #edf2f7; }
    * { box-sizing: border-box; }
    body { margin: 0; background: var(--bg); color: var(--text); font-family: 'Inter', sans-serif; overflow-x: hidden; }
    .dashboard { width: 100%; max-width: 100%; margin: 0 auto; padding: 20px; }
    .topbar { display: flex; align-items: center; justify-content: space-between; margin-bottom: 24px; background: var(--card); padding: 16px 24px; border-radius: 12px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); flex-wrap: wrap; gap: 16px; }
    .title-row { display: flex; align-items: center; gap: 24px; }
    h1 { margin: 0; font-size: 20px; font-weight: 800; color: #000; }
    .filters { display: flex; gap: 12px; flex-wrap: wrap; }
    .filter { display: flex; flex-direction: column; gap: 4px; }
    .filter label { font-size: 10px; font-weight: 700; color: var(--muted); text-transform: uppercase; }
    .filter select { border: 1px solid var(--border); background: white; padding: 6px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; }
    .kpis { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 20px; margin-bottom: 24px; }
    .kpi { background: var(--card); border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); border-top: 4px solid var(--primary); }
    .kpi-label { font-size: 12px; font-weight: 600; color: var(--muted); margin-bottom: 4px; }
    .kpi-value { font-size: 22px; font-weight: 800; color: var(--text); }
    .card { background: var(--card); border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); margin-bottom: 24px; display: flex; flex-direction: column; }
    .card h2 { margin: 0 0 15px 0; font-size: 15px; font-weight: 700; color: var(--text); }
    .grid-3 { display: grid; grid-template-columns: repeat(auto-fit, minmax(350px, 1fr)); gap: 24px; }
    .canvas-wrapper { position: relative; flex-grow: 1; min-height: 250px; min-height: 0; }
    .html-bars { display: flex; flex-direction: column; gap: 10px; }
    .bar-row { width: 100%; }
    .bar-head { display: flex; justify-content: space-between; font-size: 11px; font-weight: 700; margin-bottom: 4px; }
    .bar-bg { height: 8px; background: var(--bar-bg); border-radius: 4px; width: 100%; }
    .bar-fill { height: 100%; background: var(--primary); border-radius: 4px; }
    #catLegend { display: grid; grid-template-columns: 1fr; gap: 5px; margin-top: 15px; max-height: 100px; overflow-y: auto; }
</style>
</head>
<body>
<div class="dashboard">
    <div class="topbar">
        <div class="title-row"><h1>Strategic Market Insights (US$)</h1></div>
        <div class="filters">
            <div class="filter"><label>Year</label><select id="yearSelect"><option value="all">All Years</option></select></div>
            <div class="filter"><label>Month</label><select id="monthSelect"><option value="all">All Months</option></select></div>
            <div class="filter"><label>State Filter</label><select id="stateSelect"><option value="all">All States</option></select></div>
        </div>
    </div>
    <section class="kpis">
        <div class="kpi"><div class="kpi-label">Total Revenue</div><div class="kpi-value" id="kpi-rev">$0</div></div>
        <div class="kpi"><div class="kpi-label">Order Volume</div><div class="kpi-value" id="kpi-orders">0</div></div>
        <div class="kpi"><div class="kpi-label">Avg. Order Value</div><div class="kpi-value" id="kpi-aov">$0</div></div>
        <div class="kpi"><div class="kpi-label">Active Panelists</div><div class="kpi-value" id="kpi-users">0</div></div>
    </section>
    <section class="card" style="height: 350px;">
        <h2>Monthly Revenue Trend (US$)</h2>
        <div class="canvas-wrapper"><canvas id="trendChart"></canvas></div>
    </section>
    <div class="grid-3">
        <div class="card"><h2>Revenue by State (Top 8)</h2><div class="html-bars" id="stateBars"></div></div>
        <div class="card"><h2>Product Category Share</h2><div class="canvas-wrapper"><canvas id="catChart"></canvas></div><div id="catLegend"></div></div>
        <div class="card"><h2>Revenue by Day</h2><div class="canvas-wrapper"><canvas id="dowChart"></canvas></div></div>
    </div>
    <section class="card" style="margin-top:24px;"><h2>Top Age Segments (Revenue)</h2><div id="ageBars" class="html-bars"></div></section>
</div>
<script>
const rawData = DATA_PLACEHOLDER;
let charts = {};
const COLORS = ['#4c51bf', '#38b2ac', '#f6ad55', '#ed64a6', '#9f7aea', '#667eea', '#ecc94b', '#f56565', '#48bb78', '#a0aec0'];

function formatCurrency(v) { return new Intl.NumberFormat("en-US", { style: "currency", currency: "USD", maximumFractionDigits: 0 }).format(v || 0); }

function init() {
    if (rawData.error) return;
    const ySel = document.getElementById("yearSelect");
    const sSel = document.getElementById("stateSelect");
    rawData.years.forEach(y => { const opt = document.createElement("option"); opt.value = y; opt.textContent = y; ySel.appendChild(opt); });
    rawData.states.forEach(s => { const opt = document.createElement("option"); opt.value = s; opt.textContent = s; sSel.appendChild(opt); });
    ySel.addEventListener("change", () => { updateMonthDropdown(); updateDashboard(); });
    document.getElementById("monthSelect").addEventListener("change", updateDashboard);
    sSel.addEventListener("change", updateDashboard);
    updateDashboard();

    // Force layout recalculation after initial render
    setTimeout(() => { Object.values(charts).forEach(c => c.resize()); }, 200);
}

function updateMonthDropdown() {
    const yr = document.getElementById("yearSelect").value;
    const mSel = document.getElementById("monthSelect");
    mSel.innerHTML = '<option value="all">All Months</option>';
    if (yr !== "all" && rawData.year_month_map[yr]) {
        rawData.year_month_map[yr].forEach(m => { const opt = document.createElement("option"); opt.value = m.val; opt.textContent = m.name; mSel.appendChild(opt); });
    }
}

function updateDashboard() {
    const f = { year: document.getElementById("yearSelect").value, month: document.getElementById("monthSelect").value, state: document.getElementById("stateSelect").value };
    const match = (d) => (f.year==='all'||String(d.year)===f.year) && (f.month==='all'||String(d.month)===f.month) && (f.state==='all'||String(d.state)===f.state);

    const kpiRows = rawData.kpi_map.filter(match);
    const kpis = kpiRows.reduce((a,b) => ({rev: a.rev+b.rev, orders: a.orders+b.orders, users: Math.max(a.users, b.users)}), {rev:0, orders:0, users:0});
    document.getElementById("kpi-rev").innerText = formatCurrency(kpis.rev);
    document.getElementById("kpi-orders").innerText = kpis.orders.toLocaleString();
    document.getElementById("kpi-aov").innerText = formatCurrency(kpis.orders ? kpis.rev/kpis.orders : 0);
    document.getElementById("kpi-users").innerText = kpis.users.toLocaleString();

    updateTrend(f);
    updateStateBars(f);
    updateCategoryChart(f);
    updateDowChart(f);
    updateAgeBars(f);
}

function updateTrend(f) {
    const rows = rawData.trend_map.filter(d => (f.year==='all'||String(d.year)===f.year) && (f.state==='all'||String(d.state)===f.state));
    const agg = {}; rows.forEach(d => agg[d.month] = (agg[d.month] || 0) + d.rev);
    const values = rawData.month_labels.map((_, i) => agg[i+1] || 0);
    if (charts.trend) charts.trend.destroy();
    charts.trend = new Chart(document.getElementById("trendChart"), {
        type: 'line', data: { labels: rawData.month_labels, datasets: [{ data: values, borderColor: '#4c51bf', backgroundColor: 'rgba(76, 81, 191, 0.1)', fill: true, tension: 0.4 }] },
        options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { display: false } } }
    });
}

function updateStateBars(f) {
    const rows = rawData.kpi_map.filter(d => (f.year==='all'||String(d.year)===f.year) && (f.month==='all'||String(d.month)===f.month));
    const agg = {}; rows.forEach(d => agg[d.state] = (agg[d.state] || 0) + d.rev);
    const sorted = Object.entries(agg).sort((a,b) => b[1]-a[1]).slice(0, 8);
    const max = sorted.length ? sorted[0][1] : 1;
    document.getElementById("stateBars").innerHTML = sorted.map(([n, v]) => `
        <div class="bar-row">
            <div class="bar-head"><span>${n}</span><span>${formatCurrency(v)}</span></div>
            <div class="bar-bg"><div class="bar-fill" style="width:${(v/max)*100}%"></div></div>
        </div>`).join('');
}

function updateCategoryChart(f) {
    const rows = rawData.categories.filter(d => (f.year==='all'||String(d.year)===f.year) && (f.month==='all'||String(d.month)===f.month) && (f.state==='all'||String(d.state)===f.state));
    const agg = {}; rows.forEach(d => agg[d.Display_Category] = (agg[d.Display_Category] || 0) + d.count);
    const sorted = Object.entries(agg).sort((a,b) => b[1]-a[1]).slice(0, 10);
    const labels = sorted.map(s => s[0]); const values = sorted.map(s => s[1]);
    if (charts.cat) charts.cat.destroy();
    charts.cat = new Chart(document.getElementById("catChart"), {
        type: 'doughnut', data: { labels, datasets: [{ data: values, backgroundColor: COLORS }] },
        options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { display: false } } }
    });
    document.getElementById("catLegend").innerHTML = labels.map((l, i) => `<div style="display:flex;align-items:center;gap:6px;font-size:10px;font-weight:600;margin-bottom:2px;"><div style="width:8px;height:8px;border-radius:2px;background:${COLORS[i]}"></div>${l}</div>`).join('');
}

function updateDowChart(f) {
    const rows = rawData.dow_map.filter(d => (f.year==='all'||String(d.year)===f.year) && (f.month==='all'||String(d.month)===f.month) && (f.state==='all'||String(d.state)===f.state));
    const agg = {}; rows.forEach(d => agg[d.dow] = (agg[d.dow] || 0) + d.rev);
    if (charts.dow) charts.dow.destroy();
    charts.dow = new Chart(document.getElementById("dowChart"), {
        type: 'bar', data: { labels: rawData.day_labels.map(d=>d.slice(0,3)), datasets: [{ data: rawData.day_labels.map((_, i) => agg[i] || 0), backgroundColor: '#4c51bf' }] },
        options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { display: false } } }
    });
}

function updateAgeBars(f) {
    const rows = rawData.age.filter(d => (f.year==='all'||String(d.year)===f.year) && (f.month==='all'||String(d.month)===f.month) && (f.state==='all'||String(d.state)===f.state));
    const agg = {}; rows.forEach(d => agg[d["Q-demos-age"]] = (agg[d["Q-demos-age"]] || 0) + d["Total Price"]);
    const sorted = Object.entries(agg).sort((a,b) => b[1]-a[1]);
    const max = sorted.length ? sorted[0][1] : 1;
    document.getElementById("ageBars").innerHTML = sorted.map(([n, v]) => `
        <div class="bar-row">
            <div class="bar-head"><span>${n}</span><span>${formatCurrency(v)}</span></div>
            <div class="bar-bg"><div class="bar-fill" style="width:${(v/max)*100}%; background:#ed64a6"></div></div>
        </div>`).join('');
}

window.onerror = function(message) { google.colab.kernel.invokeFunction('report_js_error', [message], {}); };
init();
</script>
</body>
</html>
""".replace("DATA_PLACEHOLDER", dashboard_json)

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

try:
    if output: output.register_callback('report_js_error', _report_js_error)
except: pass

display(HTML(html_content))